# Capstone — When does physics emerge in CaloDiffusion?

**Read `START_HERE.md` first** (the idea) and skim `JUMPING_OFF_POINTS.md` (the plan &
deliverables). This notebook gets you to **M0** (tools working) and scaffolds the **core**
measurement — the emergence curves and lock-in steps of four observables along the x̂₀
trajectory. The physics (the observables, the lock-in, the interpretation) is left to you as
`TODO`s on purpose.

## 0 · Setup

In [ ]:
import sys, os
try:
    import calodiff_probe as cp          # works when run from this folder
except ModuleNotFoundError:              # or from the repo root
    sys.path.insert(0, 'capstone/calodiffusion')
    import calodiff_probe as cp
import numpy as np, torch, matplotlib.pyplot as plt
print('running on:', cp.DEVICE)   # 'cuda' on the GPU box, else 'cpu' (generation is slow on cpu)

## 1 · Warm-up → **M0** (worked — just run these)

Get the tools in your hands before the real work. Nothing here is graded; it de-risks the API.

### 1a · What a shower looks like
A real shower as a **layer × radius** heatmap (summed over the angular axis).

In [ ]:
model = cp.load_model('electron')
x, E, E_inc = cp.load_showers('electron', n=16)
grid = cp.regular_grid(model, x)          # (16, layer=45, angular=16, radial=9)
plt.imshow(grid[0].sum(axis=1), aspect='auto', origin='lower')
plt.xlabel('radial ring'); plt.ylabel('layer (depth)')
plt.title(f'real shower, E_inc = {E_inc[0]:.0f} GeV'); plt.colorbar(); plt.show()

### 1b · The forward (noising) process
The model learns to *reverse* this. Watch a shower dissolve into noise.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax, t in zip(axes, [0, 50, 150, 300, 399]):
    x_t, _ = cp.noise_at(model, x, t)
    ax.imshow(cp.regular_grid(model, x_t)[0].sum(1), aspect='auto', origin='lower')
    ax.set_title(f't = {t}')
plt.show()

### 1c · Does the model actually denoise?
It predicts the noise added at step `t`; the prediction should correlate with the truth (~0.9).

In [ ]:
x_t, eps = cp.noise_at(model, x, t=100)
eps_hat  = cp.predict_noise(model, x_t, E, t=100)
corr = torch.corrcoef(torch.stack([eps_hat.flatten(), eps.flatten()]))[0, 1]
print(f'corr(predicted, true noise) = {corr:.3f}   # ~0.9 = the model works')

---
## 2 · The core — emergence curves & lock-in  → **M1 / M2**

As the model builds a shower from noise, it produces at every step a running guess of the
finished shower: **x̂₀** (START_HERE §2). We compute physics on x̂₀ **in physical units** at
each step and ask: *in what order do the observables settle?*

### 2a · Get the x̂₀ trajectory (worked)
⚠️ Generation is slow on CPU — keep `n` small here; move to the **GPU** for real runs.
`x0s` is the per-step list of x̂₀ frames (index 0 = noisiest ... last = clean).

In [ ]:
n = 8
E_inc_gev = [50.] * n                     # fixed energy for the core measurement
E_cond = cp.encode_energy('electron', E_inc_gev)
x_final, xs, x0s = cp.sample(model, E_cond, debug=True)
print(f'{len(x0s)} trajectory frames, each {np.asarray(x0s[0]).shape}')
# convert one frame to physical GeV to see the shape you'll work with:
print('one frame -> physical grid:', cp.to_physical(model, x0s[-1], E_inc_gev).shape,
      '= (n, layer, angular, radial)')

### 2b · The four observables  ⟵ **your job**
Each takes a physical grid `g` of shape `(N, layer, angular, radial)` and returns one number
**per shower** (an array of length N). One is done as the pattern; write the other three.

- `total_energy` — sum of all cell energies *(worked, shows the pattern)*
- `depth` — energy-weighted **mean layer index** (how deep the shower peaks)
- `radial_spread` — energy-weighted **RMS radius** (how wide it is)
- `occupancy` — fraction of cells above a **fixed** threshold `ecut` *(use `model.config['ECUT']`)*

In [ ]:
def total_energy(g):                       # (N, layer, ang, rad) -> (N,)
    return g.sum(axis=(1, 2, 3))

def depth(g):
    # TODO: energy-weighted mean layer index.
    # hint: layer energy = g.sum(axis=(2,3)) -> (N, 45); weight arange(45) by it.
    raise NotImplementedError

def radial_spread(g):
    # TODO: energy-weighted RMS radius.
    # hint: radial energy = g.sum(axis=(1,2)) -> (N, 9); r = arange(9);
    #       mean_r = sum(r*E)/sum(E);  rms = sqrt(sum((r-mean_r)^2 * E)/sum(E)).
    raise NotImplementedError

def occupancy(g, ecut):
    # TODO: fraction of cells with energy > ecut (a number per shower).
    raise NotImplementedError

### 2c · Build the emergence curves  ⟵ **your job**
Walk the trajectory, compute each observable on x̂₀ (physical units) at every step, and
**normalise each curve to its final value** (the last frame). Then plot all four vs step.

*Expectation to test:* total energy & depth settle **early** (coarse), radial spread &
occupancy **late** (fine). Is that what you see?

In [ ]:
ecut = model.config['ECUT']
obs_fns = {'energy': total_energy, 'depth': depth,
           'radial': radial_spread, 'occupancy': lambda g: occupancy(g, ecut)}

# TODO: fill `curves[name]` with the median-over-showers value at each trajectory step.
curves = {name: [] for name in obs_fns}
for x0 in x0s:
    g = cp.to_physical(model, x0, E_inc_gev)      # (n, layer, ang, rad) in GeV
    for name, fn in obs_fns.items():
        pass  # TODO: curves[name].append( np.median(fn(g)) )

# TODO: normalise each curve to its final value, then plot vs step.
# for name, vals in curves.items():
#     vals = np.array(vals) / vals[-1]
#     plt.plot(vals, label=name)
# plt.xlabel('trajectory step (0 = noisiest)'); plt.ylabel('fraction of final value')
# plt.legend(); plt.show()

### 2d · The lock-in step + table  ⟵ **your job**
Define the **lock-in step** of an observable = the first step after which its (normalised)
curve stays within ~10% of its final value. Compute it for all four and make a small table.

*This scalar is the result* — it turns 'look at this plot' into a number you can compare
across particles (the nice-to-have).

In [ ]:
def lock_in_step(norm_curve, tol=0.1):
    # TODO: return the first index i such that |norm_curve[j] - 1| <= tol for all j >= i.
    raise NotImplementedError

# TODO: build a table: one row per observable -> its lock-in step.
# Which lock in first? Does the coarse-before-fine ordering hold?

---
## 3 · Where to go next

- **Nice-to-have (the headline):** repeat §2 for `'photon'` and `'pion'` at *matched* energies.
  Does the coarse→fine gap **widen** for the messy hadronic pion showers? → Table 2, Fig 3.
- **Optional (pick one, with us):** spectral emergence (azimuthal FFT, electron only) /
  conditioning intervention / a linear probe inside the U-Net. See `JUMPING_OFF_POINTS.md`.

Deliverables, milestones, and what's *out* of scope are all in `JUMPING_OFF_POINTS.md`.